# 4. Model track B - Decision Tree & Naive Bayes

## 4.1. Thiết lập môi trường

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent  # notebooks/ -> gốc dự án
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

FIG_DIR = project_root / "figures" / "model_track_b"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [11]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix

from src import data, metrics

## 4.2. Nạp dữ liệu đã xử lý

In [12]:
X, y, _ = data.load_processed()
fold = data.load_folds()

## 4.3. Khai báo mô hình

In [13]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=data.RANDOM_STATE),
    "Naive Bayes": GaussianNB(),
}

## 4.4. Huấn luyện & đánh giá bằng Cross-Validation

Gom thêm dự đoán out-of-fold (mỗi dòng dữ liệu được dự đoán đúng 1 lần, khi nó nằm ở tập
validation) để dùng vẽ confusion matrix ở phần sau — không rò rỉ dữ liệu vì mỗi fold luôn dự
đoán trên phần nó **chưa** thấy khi train. Bắt buộc dùng `clone(base_model)` để tạo bản sao
**chưa huấn luyện** ở mỗi fold — nếu gọi `.fit()` trực tiếp trên `base_model`, model sẽ bị "học
tiếp" nối tiếp qua các fold thay vì học lại từ đầu mỗi lần, làm sai lệch kết quả CV.

In [ ]:
rows = []
oof_true = {name: [] for name in models}
oof_pred = {name: [] for name in models}

for f, tr, va in data.iter_folds(fold):
    for name, base_model in models.items():
        model = clone(base_model) # model mới
        model.fit(X.iloc[tr], y.iloc[tr])
        proba = model.predict_proba(X.iloc[va])

        rows.append({"model": name, "fold": f, **metrics.compute_metrics(y.iloc[va], proba)})
        oof_true[name].append(y.iloc[va])
        oof_pred[name].append(proba.argmax(axis=1))

scores = pd.DataFrame(rows)
scores.head()

,model,fold,log_loss,accuracy,macro_f1
0,Decision Tree,0,8.485277,0.764583,0.568822
1,Naive Bayes,0,2.431336,0.567187,0.449300
2,Decision Tree,1,7.903322,0.780729,0.537857
3,Naive Bayes,1,1.880322,0.626563,0.485738
4,Decision Tree,2,8.504049,0.764062,0.547379


## 4.5. Lưu kết quả CV

In [15]:
scores.to_csv("../results/scores_track_b.csv", index=False)
scores

,model,fold,log_loss,accuracy,macro_f1
0,Decision Tree,0,8.485277,0.764583,0.568822
1,Naive Bayes,0,2.431336,0.567187,0.449300
2,Decision Tree,1,7.903322,0.780729,0.537857
3,Naive Bayes,1,1.880322,0.626563,0.485738
4,Decision Tree,2,8.504049,0.764062,0.547379
5,Naive Bayes,2,2.159838,0.582812,0.460278
6,Decision Tree,3,8.466504,0.765104,0.569706
7,Naive Bayes,3,2.375536,0.555729,0.446727
8,Decision Tree,4,8.184913,0.772917,0.549406
9,Naive Bayes,4,2.206390,0.612500,0.486870


## 4.6. Bảng tổng hợp kết quả (mean ± std)

In [16]:
summary = metrics.summarize_scores(scores)
summary

log_loss            accuracy            macro_f1          
                   mean       std      mean       std      mean       std
model                                                                    
Decision Tree  8.308813  0.261676  0.769479  0.007260  0.554634  0.014053
Naive Bayes    2.210685  0.216568  0.588958  0.029934  0.465783  0.019417

## 4.7. Confusion Matrix (Out-of-Fold)

Ma trận nhầm lẫn được tính trên dự đoán out-of-fold (gộp cả 5 fold), chuẩn hóa theo hàng
(`normalize="true"`) để xem tỉ lệ dự đoán đúng/nhầm của từng lớp thực tế.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, name in zip(axes, models):
    y_true_all = pd.concat(oof_true[name])
    y_pred_all = np.concatenate(oof_pred[name])
    cm = confusion_matrix(y_true_all, y_pred_all, labels=data.LABELS, normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=data.CLASS_ORDER, yticklabels=data.CLASS_ORDER, ax=ax, cbar=False)
    ax.set_title(name)
    ax.set_xlabel("Dự đoán")
    ax.set_ylabel("Thực tế")
plt.tight_layout()
fig.savefig(FIG_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 4.8. So sánh trực quan giữa 2 mô hình

Biểu đồ cột so sánh trung bình 3 chỉ số (log loss, accuracy, macro-F1) giữa Decision Tree và
Naive Bayes qua 5 fold.

In [ ]:
plot_df = summary.xs("mean", axis=1, level=1).reset_index()
plot_df = plot_df.melt(id_vars="model", var_name="metric", value_name="value")

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(data=plot_df, x="metric", y="value", hue="model", ax=ax)
ax.set_title("So sánh Decision Tree vs Naive Bayes (trung bình 5-fold)")
plt.tight_layout()
fig.savefig(FIG_DIR / "metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 4.9. Nhận xét

Qua 5-fold CV, hai mô hình cho kết quả trái ngược nhau: **Naive Bayes** tốt hơn về log loss (2.21 ±
0.22 so với 8.31 ± 0.26 của Decision Tree - chỉ số chính của bài toán), nhưng **Decision Tree**
tốt về accuracy (0.769 vs 0.589) và macro-F1 (0.555 vs 0.466).

**Nguyên nhân:** Decision Tree không giới hạn độ sâu nên xác suất dự đoán lệch cực đoan về 0/1; khi
sai, nhất là với lớp hiếm `CL` (~2.6%) - log loss bị phạt rất nặng, dù nhãn dự đoán (argmax) vẫn
đúng nhiều hơn. Naive Bayes cho xác suất "mềm" hơn nên log loss thấp, nhưng giả định đặc trưng độc
lập không phù hợp với dữ liệu xét nghiệm gan (vốn tương quan lẫn nhau) nên độ chính xác nhãn kém
hơn. Độ lệch chuẩn giữa các fold nhỏ ở cả hai model → kết quả ổn định.

**Kết luận:** theo đúng tiêu chí log loss của đề bài, **Naive Bayes** là lựa chọn tốt hơn giữa hai mô
hình Track B.